<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:**
[Write the finding from the paper.]

Methodology question:
How was the outcome or label measured, and does the validation method support this conclusion?

**Finding 2:**
[Write the finding from the paper.]

Methodology question:
Was the data split designed to prevent information leakage and support the reported result?

These questions are constructive methodology checks, similar to the validation checks applied to my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compared a normal random split with a grouped split by client. The grouped split is more realistic because the same client does not appear in both training and testing data.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Load data
url = "https://raw.githubusercontent.com/usmanumer038/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Prepare data
df = df.dropna(subset=[
    "ctr",
    "impressions_90d",
    "days_since_last_update",
    "trend_direction"
])

df["target"] = (df["trend_direction"] == "down").astype(int)

features = [
    "ctr",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

features = [f for f in features if f in df.columns]

X = df[features].fillna(0)
y = df["target"]

## Before: Random split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

random_score = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

print("Random split ROC-AUC:", round(random_score, 3))

Random split ROC-AUC: 0.707


## After: Grouped split

In [3]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model.fit(X_train, y_train)

grouped_score = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

print("Grouped split ROC-AUC:", round(grouped_score, 3))

Grouped split ROC-AUC: 0.592


# Comparison

In [4]:
results = pd.DataFrame({
    "Split": ["Random", "Grouped by Client"],
    "ROC-AUC": [random_score, grouped_score]
})

display(results.round(3))

,Split,ROC-AUC
0,Random,0.707
1,Grouped by Client,0.592


## 3. Leakage audit
I checked the features used by the model. The target and outcome-related columns are not included as features.

In [5]:
print("Features used:")
print(features)

print("\nExcluded columns:")
print([
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target"
])

print("\nLeakage check: PASSED")

Features used:
['ctr', 'impressions_90d', 'days_since_last_update', 'avg_position', 'engagement_rate', 'scroll_rate']

Excluded columns:
['content_id', 'client_id', 'trend_direction', 'trend_pct', 'target']

Leakage check: PASSED


## Error examples
The model makes incorrect predictions in some cases. Similar performance signals can lead to different outcomes. Therefore, the model should be used as decision-support, with human review before taking action.

In [6]:
predictions = model.predict(X_test)

errors = df.iloc[test_idx].copy()
errors["actual"] = y_test
errors["predicted"] = predictions

errors = errors[errors["actual"] != errors["predicted"]]

print("Incorrect predictions:", len(errors))

display(errors[features + ["actual", "predicted"]].head(5))

Incorrect predictions: 2714


,ctr,impressions_90d,days_since_last_update,avg_position,engagement_rate,scroll_rate,actual,predicted
13,0.00,307,103,39.8,0.0,25.00,0,1
19,2.02,99,20,6.9,0.0,50.00,1,0
25,0.00,27,20,7.2,0.0,0.00,1,0
26,0.12,2426,13,30.0,0.0,11.11,0,1
36,1.35,371,20,5.4,0.0,0.00,0,1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:**

The model accurately identifies content that needs to be refreshed.

**Revised claim:**

On this dataset, the model measured patterns associated with downward trends. The results are directional and should be used for decision-support rather than automatic actions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.